# Initialization

In [262]:
from qiskit import Aer, QuantumCircuit, transpile
from qiskit_ibm_provider import IBMProvider
from qiskit_ibm_runtime import QiskitRuntimeService, Session, Sampler, Estimator, Options
from qiskit_aer.noise import NoiseModel
from datetime import datetime
import mysql.connector
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import re
import sys, glob, os
import json
from enum import Enum
import time
import configparser
from datetime import datetime
import math


In [316]:
# helper function to perform sort
def num_sort(test_string):
    return list(map(int, re.findall(r'\d+', test_string)))[0]

def is_decimal_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

def is_binary_number(s):
    return all(char in '01' for char in s)

def convert_dict_binary_to_int(bin_dict):
    tmp = {}
    for key, value in bin_dict.items():
        if is_binary_number(key):
            new_key = "{}".format(int(key, 2))
            tmp[new_key] = value
    int_dict = tmp

    return int_dict

def normalize_counts(result_counts, is_json=False, shots=8192):
    if is_json:
        result_counts = json.loads(result_counts)

    result_counts = convert_dict_binary_to_int(result_counts)
    
    return {key: value / shots for key, value in result_counts.items()}

def convert_to_json(dictiontary):
    return json.dumps(dictiontary, indent = 0) 

def is_mitigated(job):
    try:
        mitigation_overhead = job.result().metadata[0]["readout_mitigation_overhead"]
        return True
    except (IndexError, KeyError):
        return False

In [317]:
from dateutil import tz
from datetime import datetime

def convert_utc_to_local(datetime_utc):
    to_zone = tz.tzlocal()

    datetime_local = datetime.fromisoformat(datetime_utc.replace('Z', '+00:00')).astimezone(to_zone)
    datetime_local = datetime_local.strftime("%Y%m%d%H%M%S")

    return datetime_local

def calculate_time_diff(time_start, time_end):
    start_datetime = datetime.fromisoformat(time_start.replace('Z', '+00:00'))
    end_datetime = datetime.fromisoformat(time_end.replace('Z', '+00:00'))
    time_difference = end_datetime - start_datetime

    return time_difference.total_seconds()    

def get_measure_lines(updated_qasm):
    lines = updated_qasm.split('\n')
    measure_lines = [line for line in lines if re.match(r'^\s*measure', line)]
    return measure_lines

def get_initial_mapping_json(updated_qasm):
    initial_mappings = []
    measure_lines = get_measure_lines(updated_qasm)
    for line in measure_lines:
        qubits = re.findall(r'q\[(\d+)\] -> c\[(\d+)\]', line)
        if len(qubits) == 1:
            initial_mappings.append((int(qubits[0][0]), int(qubits[0][1])))

    mapping = {}
    for i, j in initial_mappings:
        mapping[j] = i

    mapping_json = json.dumps(mapping, default=str)

    return mapping_json

In [1]:
# MySQL connection parameters
mysql_config = {
    'user': 'handy',
    'password': 'handy',
    'host': 'ec2-16-171-24-232.eu-north-1.compute.amazonaws.com',
    'database': 'framework'
}

In [310]:
token = "d6c68cd3c7151e9499fcaf54ff7982629e20ff25d38f32aea5b64db369985c82682f63b991dc6fc8424f4ac0349882d90a5399b03194d047b3b9b2eefb4613b3"
QiskitRuntimeService.save_account(channel="ibm_quantum", token=token, overwrite=True)
service = QiskitRuntimeService(channel="ibm_quantum", token=token)

In [311]:
backend_service = service.get_backend("ibm_brisbane")

In [256]:
# base_folder = "./circuits/adder/"
# base_folder = "./circuits/AND/"
base_folder = "./circuits/BV/"
# base_folder = "./circuits/grover/"
# base_folder = "./circuits/MQT/"
# base_folder = "./circuits/OR/"
# base_folder = "./circuits/QFT/"

resilience_level = 0
user_id = 1
runs = 2

# resilience_level = 1
# user_id = 2

qasm_files = glob.glob(os.path.expanduser(os.path.join(base_folder, "*.qasm")))
qasm_files = sorted(qasm_files)

# calling function
qasm_files.sort(key=num_sort) 

In [245]:
options = Options()
options.execution.shots = 8192
options.optimization_level = 0
options.resilience_level = resilience_level

# Sampler
sampler = Sampler(backend_service, options=options) 

# Estimator
estimator = Estimator(backend_service, options=options)

bit_format = '{0:0127b}'

# Insert to Circuit table with the correct output

In [247]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

backend_sim = Aer.get_backend('qasm_simulator')

for i, q in enumerate(qasm_files):
    qasm_source = q
    circuit_name = q.split("/")[-1].split(".")[0]

    # check if the metric is already there, just update
    cursor.execute('SELECT name FROM circuit WHERE name = %s', (circuit_name,))
    existing_row = cursor.fetchone()

    if existing_row:
        print(circuit_name, "is already exists.")

        # cursor.execute("""UPDATE circuit SET correct_output = %s WHERE name = %s""",
        # (correct_output_json, circuit_name))
        
        pass
    else:
        circuit = QuantumCircuit.from_qasm_file(qasm_source)
        qasm = circuit.qasm()
        total_gate = sum(circuit.count_ops().values())
        gates = dict(circuit.count_ops())
        depth = circuit.depth()
        
        
        job_sim = backend_sim.run(transpile(circuit, backend_sim), shots=8192)
        result_sim = job_sim.result()  
        correct_output = normalize_counts(dict(result_sim.get_counts(circuit)))
        
        gates_json = convert_to_json(gates)
        correct_output_json = convert_to_json(correct_output)
    
        cursor.execute("""INSERT INTO circuit (name, qasm, depth, total_gates, gates, correct_output)
        VALUES (%s, %s, %s, %s, %s, %s)""",
        (circuit_name, qasm, depth, total_gate, gates_json, correct_output_json))

        print(circuit_name, "inserted.")
    
conn.commit()
cursor.close()
conn.close()


bv_2 is already exists.
bv_3 is already exists.
bv_4 is already exists.
bv_5 is already exists.
bv_6 is already exists.
bv_7 is already exists.
bv_8 is already exists.
bv_9 is already exists.
bv_10 is already exists.
bv_11 is already exists.
bv_12 is already exists.
bv_13 is already exists.
bv_14 is already exists.
bv_15 is already exists.
bv_16 is already exists.


# Send to backend

## Create Header


In [248]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

now_time = datetime.now().strftime("%Y%m%d%H%M%S")

hw_name = "ibm_brisbane"

cursor.execute("""INSERT INTO result_header (user_id, hw_name, qiskit_token, created_datetime) 
VALUES (%s, %s, %s, %s)""",
(user_id, hw_name, token, now_time))
header_id = cursor.lastrowid

conn.commit()
cursor.close()
conn.close()

In [249]:
header_id

17

## Create detail

In [250]:
def insert_to_result_detail(cursor, header_id, circuit_name, compilation_name):
    now_time = datetime.now().strftime("%Y%m%d%H%M%S")
    
    sql = """
    INSERT INTO result_detail
    (header_id, circuit_name, compilation_name, created_datetime)
    VALUES (%s, %s, %s, %s);
    """

    cursor.execute(sql, (header_id, circuit_name, compilation_name, now_time))
    

In [251]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

list_circuits = []

for i, q in enumerate(qasm_files):
    qasm_source = q
    circuit_name = q.split("/")[-1].split(".")[0]

    print(circuit_name, qasm_source, i)

    circuit = QuantumCircuit.from_qasm_file(qasm_source)
    transpiled_circuit = transpile(circuit, backend_service, optimization_level=3, routing_method="sabre")

    for i in range(runs):
        list_circuits.append(transpiled_circuit)

    insert_to_result_detail(cursor, header_id, circuit_name, "qiskit_3")
    
    transpiled_circuit_na_lcd = transpile(circuit, backend_service, optimization_level=3, layout_method="noise_adaptive", routing_method="sabre")
    for i in range(runs):
        list_circuits.append(transpiled_circuit_na_lcd)

    insert_to_result_detail(cursor, header_id, circuit_name, "qiskit_NA_lcd")

conn.commit()
cursor.close()
conn.close()

bv_2 ./circuits/BV/bv_2.qasm 0
bv_3 ./circuits/BV/bv_3.qasm 1
bv_4 ./circuits/BV/bv_4.qasm 2
bv_5 ./circuits/BV/bv_5.qasm 3
bv_6 ./circuits/BV/bv_6.qasm 4
bv_7 ./circuits/BV/bv_7.qasm 5
bv_8 ./circuits/BV/bv_8.qasm 6
bv_9 ./circuits/BV/bv_9.qasm 7
bv_10 ./circuits/BV/bv_10.qasm 8
bv_11 ./circuits/BV/bv_11.qasm 9
bv_12 ./circuits/BV/bv_12.qasm 10
bv_13 ./circuits/BV/bv_13.qasm 11
bv_14 ./circuits/BV/bv_14.qasm 12
bv_15 ./circuits/BV/bv_15.qasm 13
bv_16 ./circuits/BV/bv_16.qasm 14


In [252]:
len(list_circuits)

120

### run with sampler

In [253]:
job = sampler.run(list_circuits)
job_id = job.job_id()
job_id

'cpwjsk61tcz00080eggg'

### run with estimator

In [254]:
# job = estimator.run(list_circuits)
# job_id = job.job_id()
# job_id

In [255]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

cursor.execute("""UPDATE result_header SET job_id = %s, status = "pending", updated_datetime = NOW()
WHERE id = %s """,
(job_id, header_id))

conn.commit()
cursor.close()
conn.close()

# Get Result

### Get pending status

In [258]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

cursor.execute('''SELECT distinct h.id, h.job_id, qiskit_token FROM result_header h 
INNER JOIN result_detail d ON h.id = d.header_id WHERE h.status = "pending";''')

# and h.user_id NOT IN (98, 99)
results = cursor.fetchall()

cursor.close()
conn.close()

### Get result from completed job

In [266]:
from qiskit.primitives import SamplerResult
from qiskit_ibm_runtime.utils.runner_result import RunnerResult

In [312]:
job_id = "cpwjsk61tcz00080eggg"
job = service.job(job_id)

In [318]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

cursor.execute('''SELECT d.id FROM framework.result_header h 
INNER JOIN framework.result_detail d ON h.id = d.header_id
WHERE h.status = %s AND h.job_id = %s;''', ("pending", job_id))

results_details = cursor.fetchall()

cursor.close()
conn.close()

In [319]:
if (type(job.result()) is SamplerResult):
    quasi_dists = job.result().quasi_dists

In [328]:
avg_result = {}
std_json = {}
qasm_dict = {}
mitigation_overhead_dict = {}
mitigation_time_dict = {}
no_of_optimization = len(results_details)
no_of_result = len(quasi_dists)
runs = int(no_of_result / no_of_optimization)
idx_1, idx_2 = 0, 0
shots = 8192

for idx, res in enumerate(results_details):
    detail_id = res[0]    
    avg_result[detail_id] = []
    std_json[detail_id] = []
    qasm_dict[detail_id] = []
    mitigation_overhead_dict[detail_id] = None
    mitigation_time_dict[detail_id] = None
    sum_result = {}
    std_dev = {}
    std_dict = {}

    for j in range(runs):
        res_dict = quasi_dists[idx_1]
        
        for key, value in res_dict.items():
            key_bin = bit_format.format(key)
            key_bin = key
            sum_result[key_bin] = 0
            std_dict[key_bin] = 0
            std_dev[key_bin] = []
            
        idx_1 += 1

    for j in range(runs):
        res_dict = quasi_dists[idx_2]
        
        for key, value in res_dict.items():
            key_bin = bit_format.format(key)
            key_bin = key
            sum_result[key_bin] += value
            std_dev[key_bin].append(value)
            
        idx_2 += 1
        
    for key, value in sum_result.items():
        sum_result[key] /= runs
        std_dict[key] = np.std(std_dev[key])

    avg_result[detail_id] = convert_to_json(sum_result)
    std_json[detail_id] = convert_to_json(std_dict)
    qasm_dict[detail_id] = job.inputs["circuits"][idx_2-1].qasm()

    if is_mitigated(job):
        mitigation_overhead_dict[detail_id] = job.result().metadata[idx_2-1]["readout_mitigation_overhead"]
        mitigation_time_dict[detail_id] = job.result().metadata[idx_2-1]["readout_mitigation_time"]
        
    # print(list(sum_result)[-1], sum_result[list(sum_result)[-1]])
    # print("-----------------------")

    

In [329]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

for idx, res in enumerate(results_details):
    detail_id = res[0] 
    quasi_dists = avg_result[detail_id]
    quasi_dists_std = std_json[detail_id]
    qasm = qasm_dict[detail_id]
    mapping_json = get_initial_mapping_json(qasm)
    mitigation_overhead = mitigation_overhead_dict[detail_id]
    mitigation_time = mitigation_time_dict[detail_id]

    # check if the result_backend_json is already there, just update
    cursor.execute('SELECT detail_id FROM result_backend_json WHERE detail_id = %s', (detail_id,))
    existing_row = cursor.fetchone()

    if existing_row:
        cursor.execute('''UPDATE result_backend_json SET quasi_dists = %s, quasi_dists_std = %s, qasm = %s, 
        shots = %s, mapping_json = %s, mitigation_overhead = %s, mitigation_time = %s  WHERE detail_id = %s;''',
        (quasi_dists, quasi_dists_std, qasm, shots, mapping_json, mitigation_overhead, mitigation_time, detail_id))
    else:
        cursor.execute('''INSERT INTO result_backend_json 
                           (detail_id, quasi_dists, quasi_dists_std, qasm, shots, mapping_json, mitigation_overhead, mitigation_time) 
                           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)''',
        (detail_id, quasi_dists, quasi_dists_std, qasm, shots, mapping_json, mitigation_overhead, mitigation_time))


conn.commit()
cursor.close()
conn.close()

In [330]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

execution_time = job.metrics()["usage"]["quantum_seconds"]
job_time = job.metrics()["timestamps"]
created_datetime = convert_utc_to_local(job_time["created"])
running_datetime = convert_utc_to_local(job_time["running"])
completed_datetime = convert_utc_to_local(job_time["finished"])
in_queue_second = calculate_time_diff(job_time["created"], job_time["running"])


cursor.execute("""UPDATE result_header SET status = %s, execution_time = %s, job_created_datetime = %s, 
job_in_queue_second = %s, job_running_datetime = %s, job_completed_datetime = %s, updated_datetime = NOW()  
WHERE job_id = %s""", ("executed", execution_time, created_datetime, 
                          in_queue_second, running_datetime, completed_datetime, job_id))

conn.commit()
cursor.close()
conn.close()

### Calculate the Metrics

In [331]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

cursor.execute('''SELECT j.detail_id, j.qasm, j.quasi_dists, j.quasi_dists_std FROM framework.result_backend_json j
INNER JOIN framework.result_detail d ON j.detail_id = d.id
INNER JOIN framework.result_header h ON d.header_id = h.id
WHERE h.status = %s AND h.job_id = %s;''', ("executed", job_id))

results_details_json = cursor.fetchall()

cursor.close()
conn.close()

In [332]:
def get_count_1q(qc):
    count_1q = 0
    for key, value in dict(qc.count_ops()).items():
        if key != 'cx' and key != "cy" and key != "cz" and key != "ch" and key != "crz" and key != "cp" and key != "cu" and key != "swap" and key != "ecr":
            count_1q += value

    return count_1q

def get_count_2q(qc):
    count_2q = 0
    for key, value in dict(qc.count_ops()).items():
        if key == 'cx' or key == "cy" or key == "cz" or key == "ch" or key == "crz" or key == "cp" or key == "cu" or key == "swap" or key == "ecr":
            count_2q += value

    return count_2q

def calculate_circuit_cost(qc):
    f_1q_gate = 0.8
    f_2q_gate = 0.8
    k = 0.995
    
    circuit_depth = qc.depth()
    count_1q = get_count_1q(qc)
    count_2q = get_count_2q(qc)
    
    cost = -np.log(k) * circuit_depth - np.log(f_1q_gate) * count_1q - np.log(f_2q_gate) * count_2q

    return cost

def get_correct_output_dict(cursor, detail_id):
    cursor.execute('''SELECT c.correct_output FROM framework.result_detail d
    INNER JOIN framework.circuit c ON d.circuit_name = c.name
    WHERE d.id = %s;''', (detail_id, ))
    
    result_correct = cursor.fetchall()

    correct_output = json.loads(result_correct[0][0])

    return correct_output

def calculate_success_rate_nassc(correct_output, dists):
    success_rate = 0
    for key, value in dists.items():
        if key in correct_output:
            success_rate = success_rate + value

    return success_rate

def calculate_success_rate_tvd(correct_output, dists):
    sr_aux = 0
    for key, value in dists.items():
        if key in correct_output:
            sr_aux = sr_aux + abs(correct_output[key] - value)
        else: 
            sr_aux = sr_aux + value
    tvd = sr_aux / 2

    return 1 - tvd

def calculate_hellinger_distance(correct_output, dists):
    hd_aux = 0
    for key, value in dists.items():
        if key in correct_output:
            if value < 0:
                value = 0
            hd_aux = hd_aux + (math.sqrt(correct_output[key]) - math.sqrt(value))**2
        else: 
            hd_aux = hd_aux + value

    if hd_aux < 0:
        hd_aux = 0

    hellinger_distance = math.sqrt(hd_aux)/math.sqrt(2)

    return hellinger_distance


In [333]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

for idx, res in enumerate(results_details_json):
    detail_id, qasm, quasi_dists, quasi_dists_std = res

    quasi_dists_dict = json.loads(quasi_dists) 
    quasi_dists_std_dict = json.loads(quasi_dists_std) 
    
    qc = QuantumCircuit.from_qasm_str(qasm)
    total_gate = sum(qc.count_ops().values())
    total_one_qubit_gate = get_count_1q(qc)
    total_two_qubit_gate = get_count_2q(qc)
    circuit_depth = qc.depth()
    circuit_cost = calculate_circuit_cost(qc)

    correct_output = get_correct_output_dict(cursor, detail_id)
    success_rate_quasi = calculate_success_rate_nassc(correct_output, quasi_dists_dict)
    success_rate_nassc = success_rate_quasi
    success_rate_quasi_std = calculate_success_rate_nassc(correct_output, quasi_dists_std_dict)
    success_rate_tvd = calculate_success_rate_tvd(correct_output, quasi_dists_dict)
    hellinger_distance = calculate_hellinger_distance(correct_output, quasi_dists_dict)

    # check if the metric is already there, just update
    cursor.execute('SELECT detail_id FROM metric WHERE detail_id = %s', (detail_id,))
    existing_row = cursor.fetchone()

    if existing_row:
        cursor.execute("""UPDATE metric SET total_gate = %s, total_one_qubit_gate = %s, total_two_qubit_gate = %s, circuit_depth = %s, 
        circuit_cost = %s, success_rate_tvd = %s, success_rate_nassc = %s, success_rate_quasi = %s, hellinger_distance = %s
        WHERE detail_id = %s; """, 
        (total_gate, total_one_qubit_gate, total_two_qubit_gate, circuit_depth, 
         circuit_cost, success_rate_tvd, success_rate_nassc, success_rate_quasi, hellinger_distance, detail_id))
    else:
        cursor.execute("""INSERT INTO metric(detail_id, total_gate, total_one_qubit_gate, total_two_qubit_gate, circuit_depth, 
        circuit_cost, success_rate_tvd, success_rate_nassc, success_rate_quasi, hellinger_distance)
        VALUES (%s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s); """, 
        (detail_id, total_gate, total_one_qubit_gate, total_two_qubit_gate, circuit_depth, 
         circuit_cost, success_rate_tvd, success_rate_nassc, success_rate_quasi, hellinger_distance))

conn.commit()
cursor.close()
conn.close()

In [334]:
conn = mysql.connector.connect(**mysql_config)
cursor = conn.cursor()

cursor.execute("""UPDATE result_header SET status = %s, updated_datetime = NOW()  
WHERE job_id = %s""", ("done", job_id))

conn.commit()
cursor.close()
conn.close()

# add measurements to circuits

In [252]:
# base_folder = "./circuits/QFT/"

# qasm_files = glob.glob(os.path.expanduser(os.path.join(base_folder, "*.qasm")))
# qasm_files = sorted(qasm_files)

# # calling function
# qasm_files.sort(key=num_sort) 

# for i, q in enumerate(qasm_files):
#     qasm_source = q
#     circuit_name = q.split("/")[-1].split(".")[0]
#     no_q = int(circuit_name.split("_")[1])
#     print(circuit_name, no_q)
    

In [253]:
# for i, q in enumerate(qasm_files):
#     qasm_source = q
#     circuit_name = q.split("/")[-1].split(".")[0]
#     no_q = int(circuit_name.split("_")[1])
#     print(circuit_name, no_q)

#     with open(qasm_source, 'a') as file:
#         for j in range(no_q):
#             file.write("measure q[{}] -> c[{}];\n".format(j,j))
